## EDGE Model Ver. B

### This notebook will contain a variant of the Rocket Richard Baseline Predictor made in rr1.ipynb, that is, this variant will be using General Skater Stats (GSS) + EDGE Stats -- AKA: EDGE Model Ver. B
#### This is done for the sake of model fine tuning and evaluating performance

In [236]:
import numpy as np
import pandas as pd
from nhlpy import NHLClient
import ast
from helpersrr import clear_csv, extractPlayerID, placeToStats, fetchSkaterStats, labelWinners, formatEdgeStats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier


import importlib
import helpersrr
importlib.reload(helpersrr)

<module 'helpersrr' from '/Users/Joaquin/Documents/GitHub/nhl-trophy-predictor/notebooks/helpersrr.py'>

In [237]:
#global variables
ranks = True
versionA = True    #version A has no shot details, version B has ALL shot details

client = NHLClient()
rocket_richard = clear_csv("../data/formattedwebscraped/rocketrichard.csv")

first_place = rocket_richard[['szn','winner']]
first_ids = placeToStats(first_place)
second_place = rocket_richard[['szn', 'runner_up']]
second_ids = placeToStats(second_place)
third_place = rocket_richard[['szn', 'finalist']]
third_ids = placeToStats(third_place)

../data/formattedwebscraped/rocketrichard.csv


### Milestone 1: Get EDGE Statistics from the API and place them in data/api/EDGEstats

In [238]:
#DO NOT RUN THIS IF COMMENTED -- essentially places relevant EDGE stats files into the data/api/EDGEstats folder
#edgeSzns = ['20212022','20222023','20232024','20242025','20252026']
#fetchSkaterStats('20212022',csv=True,edge=True)
#for season in edgeSzns:
#    fetchSkaterStats(season, csv=True, edge=True)
#    print(f"done season {season}")

### Milestone 2: Make the new feature set (GSS + EDGE Stats)

In [239]:
#get the playerIds of the top 3
rr2firstids = placeToStats(rocket_richard[['szn','winner']])
rr2secondids = placeToStats(rocket_richard[['szn','runner_up']])
rr2thirdids = placeToStats(rocket_richard[['szn','finalist']])


In [240]:
#split training sets and testing set
edgeSzns = ['20212022','20222023','20232024','20242025','20252026']
trainingSets = []
testingSet = []

to_drop = ['positionCode','lastName','teamAbbrevs','shGoals','shPoints']    #keep full name instead of last name
for year in edgeSzns:
    if ranks == True:
        #if versionA == True:
        #    modifiedDf = labelWinners(year=year[:4], first_ids=rr2firstids, second_ids=rr2secondids, third_ids=rr2thirdids, rank=True, edge=True, versionA=True)
            #print(modifiedDf.columns)
        #else:
        modifiedDf = labelWinners(year=year[:4], first_ids=rr2firstids, second_ids=rr2secondids, third_ids=rr2thirdids, rank=True, edge=True, versionA=False)
    else:
        #if versionA == True:
        #    modifiedDf = labelWinners(year=year[:4], first_ids=rr2firstids, second_ids=rr2secondids, third_ids=rr2thirdids, rank=False, edge=True, versionA=True)
            #print(modifiedDf.columns)
        #else:
        modifiedDf = labelWinners(year=year[:4], first_ids=rr2firstids, second_ids=rr2secondids, third_ids=rr2thirdids, rank=False, edge=True, versionA=False)
    
    feature_df = modifiedDf.drop(columns=to_drop)
    feature_df = feature_df.dropna()
    feature_df .loc[feature_df ['shootsCatches'] == "L", 'shootsCatches'] = 0    #encode L -> 0 to fit model
    feature_df .loc[feature_df ['shootsCatches'] == "R", 'shootsCatches'] = 1    #encode R -> 1 to fit model
    
    if year == "20252026":  #2025-2026 will be the testing year
        testingSet.append(feature_df)
    else:
        trainingSets.append(feature_df)
print(len(testingSet), len(trainingSets))

masterTraining = pd.DataFrame()
for train in trainingSets:
    masterTraining = pd.concat([masterTraining,train])

masterTesting = pd.DataFrame()
masterTesting = pd.concat([masterTesting,testingSet[0]])

if versionA == True:
    masterTraining = masterTraining.drop(columns=[
                'Behind the Net Shots',
                        'Beyond Red Line Shots',
                        'Center Point Shots',
                        'Crease Shots',
                        'High Slot Shots',
                        'L Circle Shots',
                        'L Corner Shots',
                        'L Net Side Shots',
                        'L Point Shots',
                        'Low Slot Shots',
                        'Offensive Neutral Zone Shots',
                        'Outside L Shots',
                        'Outside R Shots',
                        'R Circle Shots',
                        'R Corner Shots',
                        'R Net Side Shots',
                        'R Point Shots'
                        ])
    masterTesting = masterTesting.drop(columns=[
                'Behind the Net Shots',
                        'Beyond Red Line Shots',
                        'Center Point Shots',
                        'Crease Shots',
                        'High Slot Shots',
                        'L Circle Shots',
                        'L Corner Shots',
                        'L Net Side Shots',
                        'L Point Shots',
                        'Low Slot Shots',
                        'Offensive Neutral Zone Shots',
                        'Outside L Shots',
                        'Outside R Shots',
                        'R Circle Shots',
                        'R Corner Shots',
                        'R Net Side Shots',
                        'R Point Shots'
                        ])

1 4


In [241]:
#with pd.option_context("display.max_rows", None, "display.max_columns", None):      #view full prediction list
#    display(masterTraining)  

### Milestone 3: Train model

In [242]:
if ranks == False:
    train_x = masterTraining.drop(columns=['skaterFullName','rrWinner','playerId','seasonId'])
    train_y = masterTraining['rrWinner']
else:
    train_x = masterTesting.drop(columns=['skaterFullName','rrRank','playerId','seasonId'])
    train_y = masterTesting['rrRank']

model1 = LogisticRegression(max_iter=20000, class_weight="balanced")
model1.fit(train_x,train_y)

#new addition: RandomForest
#model2 = RandomForestClassifier(n_estimators=100)
#model2.fit(train_x,train_y)

#new addition: Gradient Boosting; w/ default parameters, but just written for clarification
model3 = GradientBoostingClassifier(loss='log_loss', learning_rate=0.1) 
model3.fit(train_x, train_y)


,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"init init: estimator or 'zero', default=NoneAn estimator object that is used to co

### Milestone 4: Test on EDGE + GSS of 2025-2026

In [243]:
test2025 = masterTesting
if ranks == False:
    test_x = test2025.drop(columns=['skaterFullName','rrWinner','playerId','seasonId'])
    test_y = test2025['rrWinner']
else:
    test_x = test2025.drop(columns=['skaterFullName','rrRank','playerId','seasonId'])
    test_y = test2025['rrRank']


pred_y1 = model1.predict(test_x)
predictions = pd.Series(pred_y1)
predictions = predictions.rename('predictions')
predictions = predictions.to_frame()
results = test2025.join(predictions)

'''
pred_y2 = model2.predict(test_x)
predictions2 = pd.Series(pred_y2)
predictions2 = predictions2.rename('predictions')
predictions2 = predictions2.to_frame()
results2 = test2025.join(predictions2)
'''

pred_y3 = model3.predict(test_x)
predictions3 = pd.Series(pred_y3)
predictions3 = predictions3.rename('predictions')
predictions3 = predictions3.to_frame()
results3 = test2025.join(predictions3)

In [244]:
#for model 1

if ranks == False:
    show = results.loc[results['predictions'] == 1]
else:
    show = results.loc[results['predictions'] != 0]
    show = show.dropna()
    hyman = results.loc[results['rrRank'] != 0]

#view influences on the model
feature_names = train_x.columns
coefficients = pd.Series(model1.coef_[0],index=feature_names)
print(coefficients.sort_values())
print(masterTraining.shape)

with pd.option_context("display.max_rows", None, "display.max_columns", None):      #view full prediction list
    display(show)  


plusMinus             -0.105485
evGoals               -0.083761
shots                 -0.078498
goals                 -0.077283
points                -0.073356
totalDistanceSkated   -0.051901
evPoints              -0.045396
midGoals              -0.039082
gameWinningGoals      -0.029298
ppPoints              -0.027603
otGoals               -0.020556
highGoals             -0.019100
highShots             -0.007585
shootsCatches         -0.003403
pointsPerGame         -0.000882
shootingPct           -0.000217
distanceMaxGame       -0.000071
offensiveZonePctg     -0.000062
faceoffWinPct          0.000030
neutralZonePctg        0.000061
defensiveZonePctg      0.000213
averageTOI             0.000704
assists                0.003927
skatingSpeed           0.004167
ppGoals                0.006363
gamesPlayed            0.007151
longGoals              0.010723
timeOnIcePerGame       0.042225
topShotSpeed           0.053338
midShots               0.053633
penaltyMinutes         0.061164
longShot

,assists,evGoals,evPoints,faceoffWinPct,gameWinningGoals,gamesPlayed,goals,otGoals,penaltyMinutes,playerId,plusMinus,points,pointsPerGame,ppGoals,ppPoints,seasonId,shootingPct,shootsCatches,shots,skaterFullName,timeOnIcePerGame,topShotSpeed,skatingSpeed,totalDistanceSkated,distanceMaxGame,longShots,longGoals,midShots,midGoals,highShots,highGoals,offensiveZonePctg,neutralZonePctg,defensiveZonePctg,averageTOI,rrRank,predictions
0,90,34,82,0.49512,4,82,48,1,44,8478402,17,138,1.68292,13,54,20252026,0.15686,0,306,Connor McDavid,1379.1219,132.0467,39.6089,531.4875,7.9080,8,1,92,9,120,26,0.476879,0.169191,0.353930,22.985365,3.0,3.0
2,74,42,97,0.51049,7,80,53,1,39,8477492,57,127,1.58750,11,30,20252026,0.15142,1,350,Nathan MacKinnon,1335.7125,146.4342,39.8786,456.0059,7.8713,32,6,149,20,83,19,0.466145,0.180970,0.352885,22.261875,1.0,1.0
14,50,28,56,0.47826,5,81,38,2,61,8477404,13,88,1.08641,8,30,20252026,0.17272,0,220,Jake Guentzel,1214.7530,133.6560,35.8189,405.7720,6.8168,10,0,62,8,116,26,0.439654,0.177980,0.382366,20.245883,0.0,2.0


findings/conclusions on the above can be found in commitlog.md

In [245]:
#for model 2 -- uses SHAP to visualize predictions (later)
if ranks == False:
    show = results3.loc[results3['predictions'] == 1]
else:
    show = results3.loc[results3['predictions'] != 0]
    show = show.dropna()
    hyman = results3.loc[results3['rrRank'] != 0]

with pd.option_context("display.max_rows", None, "display.max_columns", None):      #view full prediction list
    display(show) 

,assists,evGoals,evPoints,faceoffWinPct,gameWinningGoals,gamesPlayed,goals,otGoals,penaltyMinutes,playerId,plusMinus,points,pointsPerGame,ppGoals,ppPoints,seasonId,shootingPct,shootsCatches,shots,skaterFullName,timeOnIcePerGame,topShotSpeed,skatingSpeed,totalDistanceSkated,distanceMaxGame,longShots,longGoals,midShots,midGoals,highShots,highGoals,offensiveZonePctg,neutralZonePctg,defensiveZonePctg,averageTOI,rrRank,predictions
0,90,34,82,0.49512,4,82,48,1,44,8478402,17,138,1.68292,13,54,20252026,0.15686,0,306,Connor McDavid,1379.1219,132.0467,39.6089,531.4875,7.9080,8,1,92,9,120,26,0.476879,0.169191,0.353930,22.985365,3.0,3.0
2,74,42,97,0.51049,7,80,53,1,39,8477492,57,127,1.58750,11,30,20252026,0.15142,1,350,Nathan MacKinnon,1335.7125,146.4342,39.8786,456.0059,7.8713,32,6,149,20,83,19,0.466145,0.180970,0.352885,22.261875,1.0,1.0
14,50,28,56,0.47826,5,81,38,2,61,8477404,13,88,1.08641,8,30,20252026,0.17272,0,220,Jake Guentzel,1214.7530,133.6560,35.8189,405.7720,6.8168,10,0,62,8,116,26,0.439654,0.177980,0.382366,20.245883,0.0,2.0
